## Assumptions:
- Invalid timestamps are rejected.
- Missing active_power or setpoint values are rejected.
- Duplicate records are removed.
- Numeric values may contain thousands separators.
- Bad rows are logged rather than causing the pipeline to fail.

In [40]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path("../task_2_assets/data/telemetry_raw.csv")
CLEAN_PATH = Path("../task_2_assets/data/telemetry_cleaned.csv")

In [41]:
raw = pd.read_csv(RAW_PATH, thousands=",")

print(f"{len(raw)} rows loaded from {RAW_PATH}")
raw

12 rows loaded from ..\task_2_assets\data\telemetry_raw.csv


,timestamp,active_power,setpoint,site_id
0,01/07/2026 00:00:00,18200.0,18000.0,1
1,01/07/2026 00:30:00,18450.0,18100.0,1
2,01/07/2026 01:00:00,12500.0,18300.0,1
3,01/07/2026 01:30:00,19000.0,NaN,1
4,01/07/2026 02:00:00,19100.0,18800.0,1
5,01/07/2026 02:30:00,19350.0,18950.0,1
6,01/07/2026 02:30:00,19350.0,18950.0,1
7,01/07/2026 03:00:00,NaN,19100.0,1
8,01/07/2026 03:30:00,19700.0,19250.0,1
9,01/07/2026 04:00:00,19850.0,19300.0,1


In [42]:
df = raw.copy()

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

df["active_power"] = pd.to_numeric(df["active_power"], errors="coerce")
df["setpoint"] = pd.to_numeric(df["setpoint"], errors="coerce")
df["site_id"] = pd.to_numeric(df["site_id"], errors="coerce")

df

,timestamp,active_power,setpoint,site_id
0,2026-07-01 00:00:00,18200.0,18000.0,1
1,2026-07-01 00:30:00,18450.0,18100.0,1
2,2026-07-01 01:00:00,12500.0,18300.0,1
3,2026-07-01 01:30:00,19000.0,NaN,1
4,2026-07-01 02:00:00,19100.0,18800.0,1
5,2026-07-01 02:30:00,19350.0,18950.0,1
6,2026-07-01 02:30:00,19350.0,18950.0,1
7,2026-07-01 03:00:00,NaN,19100.0,1
8,2026-07-01 03:30:00,19700.0,19250.0,1
9,2026-07-01 04:00:00,19850.0,19300.0,1


In [43]:
def validate(frame, duplicate_keep):
    return {
        "no duplicate rows": ~frame.duplicated(keep=duplicate_keep),
        "has timestamp": frame["timestamp"].notna(),
        "has active_power": frame["active_power"].notna(),
        "has setpoint": frame["setpoint"].notna(),
        "has site_id": frame["site_id"].notna(),
    }


def report_validation_results(rules):
    all_passed = True

    for rule_name, passed in rules.items():
        fail_count = (~passed).sum()
        status = "PASS" if fail_count == 0 else "FAIL"
        all_passed = all_passed and fail_count == 0

        print(f"[{status}] {rule_name}: {fail_count} row(s) failed")

    return all_passed

In [44]:
print("Validation results before cleaning:")

pre_clean_rules = validate(df, duplicate_keep="first")
report_validation_results(pre_clean_rules)

Validation results before cleaning:
[FAIL] no duplicate rows: 1 row(s) failed
[FAIL] has timestamp: 1 row(s) failed
[FAIL] has active_power: 1 row(s) failed
[FAIL] has setpoint: 1 row(s) failed
[PASS] has site_id: 0 row(s) failed


np.False_

In [45]:
validation_results = pd.concat(pre_clean_rules, axis=1)

invalid_rows = df[~validation_results.all(axis=1)]

print(f"{len(invalid_rows)} invalid row(s) found")
invalid_rows

4 invalid row(s) found


,timestamp,active_power,setpoint,site_id
3,2026-07-01 01:30:00,19000.0,NaN,1
6,2026-07-01 02:30:00,19350.0,18950.0,1
7,2026-07-01 03:00:00,NaN,19100.0,1
10,NaT,20000.0,19400.0,1


In [46]:
valid_rows = validation_results.all(axis=1)

clean = (
    df[valid_rows]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print(f"{len(df)} rows in")
print(f"{len(clean)} valid rows kept")
print(f"{(~valid_rows).sum()} invalid rows dropped")

clean

12 rows in
8 valid rows kept
4 invalid rows dropped


,timestamp,active_power,setpoint,site_id
0,2026-07-01 00:00:00,18200.0,18000.0,1
1,2026-07-01 00:30:00,18450.0,18100.0,1
2,2026-07-01 01:00:00,12500.0,18300.0,1
3,2026-07-01 02:00:00,19100.0,18800.0,1
4,2026-07-01 02:30:00,19350.0,18950.0,1
5,2026-07-01 03:30:00,19700.0,19250.0,1
6,2026-07-01 04:00:00,19850.0,19300.0,1
7,2026-07-01 04:30:00,20500.0,19550.0,1


In [47]:
print("Validation results after cleaning:")

post_clean_rules = validate(clean, duplicate_keep=False)
clean_is_valid = report_validation_results(post_clean_rules)

assert clean_is_valid, "Cleaned data failed validation and is not safe to load"

print("All validation checks passed. Data is safe to load.")

Validation results after cleaning:
[PASS] no duplicate rows: 0 row(s) failed
[PASS] has timestamp: 0 row(s) failed
[PASS] has active_power: 0 row(s) failed
[PASS] has setpoint: 0 row(s) failed
[PASS] has site_id: 0 row(s) failed
All validation checks passed. Data is safe to load.


In [48]:
clean.to_csv(CLEAN_PATH, index=False)

print(f"Saved {len(clean)} clean rows to {CLEAN_PATH}")

Saved 8 clean rows to ..\task_2_assets\data\telemetry_cleaned.csv


## Summary

The raw CSV was loaded using pandas.

The data was then cleaned by:
- thousands separators handled during import using `thousands=","`.
- parsing timestamps using expected `DD/MM/YYYY HH:MM:SS` format
- converting numeric fields to numeric types
- coercing invalid values to nulls rather than allowing the notebook to fail
- validating required fields
- identifying and removing duplicate rows
- explicitly reporting validation failures before and after cleaning

Rows were only kept if they passed all validation rules. The cleaned output was re-validated before being saved to `telemetry_cleaned.csv`.